# CQD-SHAP Colab Resume From Saved Shapley Results

Use this notebook when Colab/runtime was reset, but you already downloaded these two files to your computer:

- `bench1_2p_shapley_necessary.csv`
- `bench1_2p_shapley_sufficient.csv`

This notebook restores those files into the correct Colab folder and runs only the missing baseline methods: `score`, `random`, `first`, and `last`.

## 1. Runtime Check

Recommended: `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone CQD-SHAP and Install Helpers

In [ ]:
%cd /content
!rm -rf CQD-SHAP
!git clone https://github.com/ds-jrg/CQD-SHAP.git
%cd /content/CQD-SHAP
!pip -q install gdown pandas networkx matplotlib tqdm

## 3. Download Data and Models

The saved CSV files are only results. To continue running baselines, Colab still needs the data and model files.

In [ ]:
%cd /content/CQD-SHAP

!test -f data.zip || gdown 1yoZFUAY7DLOj4fC78pIU32SUSAEWRmLw -O data.zip
!test -d data || unzip -q data.zip

!test -f models.zip || gdown 1ot3CuVk4DorVu3JiHKzdumzGNaTREAU3 -O models.zip
!test -d models || unzip -q models.zip

## 4. Upload Your Saved Shapley CSV Files

When this cell asks for files, upload exactly these two files from your computer:

- `bench1_2p_shapley_necessary.csv`
- `bench1_2p_shapley_sufficient.csv`

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()

output_dir = Path("evaluation_benchmark1/Freebase")
output_dir.mkdir(parents=True, exist_ok=True)

expected_files = [
    "bench1_2p_shapley_necessary.csv",
    "bench1_2p_shapley_sufficient.csv",
]

for filename in expected_files:
    if filename not in uploaded:
        raise FileNotFoundError(f"Please upload {filename}")
    shutil.move(filename, output_dir / filename)

print("Restored saved files:")
!find evaluation_benchmark1/Freebase -maxdepth 1 -type f | sort

## 5. Create Limited Evaluator

This creates `evaluation_limited.py` with `--max_queries`, so baseline runs match your limited experiment setup.

In [ ]:
from pathlib import Path

src_path = Path("evaluation.py")
limited_path = Path("evaluation_limited.py")
src = src_path.read_text()

if "--max_queries" not in src:
    src = src.replace(
        "parser.add_argument('--normalize', action='store_true', help='Whether to normalize scores using sigmoid')",
        "parser.add_argument('--normalize', action='store_true', help='Whether to normalize scores using sigmoid')\n"
        "    parser.add_argument('--max_queries', type=int, help='Limit number of queries per query type')"
    )
    src = src.replace(
        "        hard = query_dataset_hard.get_queries(query_type)\n"
        "        complete = query_dataset.get_queries(query_type)\n",
        "        hard = query_dataset_hard.get_queries(query_type)\n"
        "        complete = query_dataset.get_queries(query_type)\n"
        "        if args.max_queries is not None:\n"
        "            hard = hard[:args.max_queries]\n"
        "            complete = complete[:args.max_queries]\n"
        "            logging.info(f'Limited {query_type} to {len(hard)} queries')\n"
    )

limited_path.write_text(src)
print("Created", limited_path)
!python evaluation_limited.py --help | grep max_queries

## 6. Run Missing Baselines

This keeps your uploaded `shapley` files and runs only `score`, `random`, `first`, and `last` for `2p`.

In [ ]:
import subprocess
from pathlib import Path

MAX_QUERIES = 2000
benchmark_version = 1
query_type = "2p"
data_dir = "data/FB15k-237"
model_path = "models/FB15k-237-model-rank-1000-epoch-100-1602508358.pt"
output_dir = Path("evaluation_benchmark1/Freebase")

methods = ["score", "random", "first", "last"]

for method in methods:
    necessary_file = output_dir / f"bench1_2p_{method}_necessary.csv"
    sufficient_file = output_dir / f"bench1_2p_{method}_sufficient.csv"

    if necessary_file.exists() and sufficient_file.exists():
        print(f"SKIP {method}: outputs already exist")
        continue

    print(f"RUN {method}")
    subprocess.run([
        "python", "evaluation_limited.py",
        "--kg", "Freebase",
        "--benchmark", str(benchmark_version),
        "--query_type", query_type,
        "--method", method,
        "--data_dir", data_dir,
        "--model_path", model_path,
        "--max_queries", str(MAX_QUERIES),
    ], check=True)

## 7. Summarize All Results

In [ ]:
from pathlib import Path
import pandas as pd

files = sorted(Path("evaluation_benchmark1/Freebase").glob("bench1_2p_*_*.csv"))
print("CSV files found:", len(files))
for path in files:
    print("-", path)

rows = []
for path in files:
    df = pd.read_csv(path)
    filename = path.name
    method = next((m for m in ["shapley", "score", "random", "first", "last"] if f"_{m}_" in filename), "unknown")
    scenario = "necessary" if "necessary" in filename else "sufficient"
    rows.append({
        "method": method,
        "scenario": scenario,
        "file": filename,
        "rows/question_answer_cases": len(df),
        "unique_queries": df["query_idx"].nunique() if "query_idx" in df.columns else None,
        "mean_delta_mrr": df["delta_mrr"].mean() if "delta_mrr" in df.columns else None,
        "mean_delta_hit_1": df["delta_hit_1"].mean() if "delta_hit_1" in df.columns else None,
        "mean_runtime": df["runtime"].mean() if "runtime" in df.columns else None,
    })

summary = pd.DataFrame(rows).sort_values(["method", "scenario"])
summary

## 8. Save and Download Outputs

In [ ]:
summary.to_csv("evaluation_summary_resumed_2p.csv", index=False)
!zip -r cqd_shap_resumed_outputs.zip evaluation_benchmark1/Freebase evaluation_summary_resumed_2p.csv

from google.colab import files
files.download("cqd_shap_resumed_outputs.zip")